# Networking — VNets, Peering, VPN & ExpressRoute

Identity decides *who* can call your services. Networking decides *which packets ever get to try*. Even on the most carefully RBAC'd subscription, a misconfigured network is the path most incidents take — a public IP that shouldn't exist, a route that bypasses the firewall, a subnet that leaks into the wrong VNet. Azure's networking surface is wide, but it composes from a small set of primitives, and once you have the shape in your head the rest is configuration.

The mental model: **VNet** is your address space, **subnets** carve it up, **NSGs** are the firewall on each subnet, **routes** decide where packets go, and **peering, VPN, ExpressRoute, Private Link, and Virtual WAN** are the various ways a VNet talks to other VNets, to on-prem, and to Azure PaaS.

## Virtual Networks and subnets

A **Virtual Network (VNet)** is a private RFC 1918 IPv4 (and optional IPv6) address space scoped to a single region and subscription. You pick a CIDR block at create — say `10.50.0.0/16` — and carve **subnets** out of it. Resources you place in a subnet get a NIC with a private IP from that subnet's range.

Three properties of subnets you cannot un-pick later without pain:

- **Size.** Azure reserves five IPs per subnet (`.0`, `.1`, `.2`, `.3`, `.255` on a /24). Sizes you find under-spec'd six months in are expensive to grow.
- **Delegation.** Some PaaS services (App Service VNet integration, Container Apps, Azure Database for PostgreSQL Flexible Server) require a *delegated* subnet they own exclusively. Once delegated, no other resource can join it.
- **Service association.** A subnet can hold either NICs or a specific service like an Azure Bastion or VPN Gateway — never both.

Address-space planning matters more than it looks. VNets that need to peer must have **non-overlapping** ranges. If every team grabs `10.0.0.0/16`, your peering story dies on day one. Plan a single org-wide RFC 1918 allocation up front — a /14 or /13 you carve `/16`s out of per workload — and stick to it.

AWS comparison: VNet ≈ VPC; subnet ≈ subnet (Azure subnets are *not* zonal — a subnet spans the region, and zone placement is per-resource).

## NSGs and Application Security Groups

A **Network Security Group (NSG)** is a stateful firewall you attach to a subnet, a NIC, or both. Rules are five-tuple matches (source, source port, destination, destination port, protocol) with `Allow` or `Deny`, evaluated in priority order. Lowest number wins.

Three rule sources that surprise newcomers:

- **Service tags** — symbolic names for Microsoft IP ranges (`Storage`, `AzureKeyVault`, `Sql`, `Internet`, `VirtualNetwork`). Microsoft updates the underlying ranges; you don't.
- **Default rules** — every NSG ships with three allow + three deny rules at the bottom of the list (AllowVnetInBound, AllowAzureLoadBalancerInBound, DenyAllInBound, plus outbound counterparts). You cannot delete them; you can only override them with higher-priority rules.
- **Effective rules** — when an NSG is attached at both the NIC and the subnet, *both* must allow the traffic. Use the "effective security rules" view to see what's actually happening.

**Application Security Groups (ASGs)** let you express NSG rules against logical groups of NICs rather than IP ranges. Tag the NICs of every web tier VM as `asg-web`, every database VM as `asg-db`, and write rules like "allow `asg-web` to `asg-db` on 5432". When the cluster scales, new NICs inherit the tag and the rule keeps working without IP juggling.

AWS comparison: NSG ≈ AWS security group + NACL combined; ASG ≈ AWS security-group-referencing-security-group pattern, but as a first-class object.

## Routes — system, user-defined, and BGP

Every subnet has a routing table. Three sources of routes, in increasing priority:

- **System routes** — Azure auto-creates routes for the VNet, peered VNets, gateways, and the default `0.0.0.0/0 → Internet`.
- **User-defined routes (UDRs)** — you write a **route table**, associate it with a subnet, and override system routes. Common UDR pattern: `0.0.0.0/0 → next hop: Azure Firewall` to force all egress through a firewall.
- **BGP routes** — learned dynamically from a VPN Gateway or ExpressRoute. Used to advertise on-prem networks into Azure and vice versa.

Routing in Azure is **longest-prefix-match**, then route-source priority (UDR > BGP > system). A common bug is forgetting that when you turn on **forced tunneling** via a UDR pointing at a firewall, you also need a route *back* for response packets — without it, the firewall sees an asymmetric flow and drops it.

The "effective routes" view on a NIC is your friend. Open it whenever a packet isn't going where you expected.

## Egress — Azure Firewall, NVA, NAT Gateway

Three ways for VMs in a private subnet to reach the internet:

- **Public IP on each VM** — simplest, least secure, hardest to audit. Avoid for production.
- **NAT Gateway** — a managed, stateful, outbound-only NAT. Attach to a subnet; all egress from that subnet leaves through the NAT Gateway's pool of public IPs. Scales transparently, charges per GB processed. The right default for outbound-only traffic that doesn't need L7 inspection.
- **Azure Firewall** — a managed L3–L7 firewall with FQDN filtering, threat intelligence feed, IDPS, and TLS inspection (Premium tier). Goes in a dedicated `AzureFirewallSubnet`, receives traffic via UDR. Three tiers (Basic / Standard / Premium); Premium is what most enterprises actually run.
- **Network Virtual Appliance (NVA)** — third-party firewall image (Palo Alto, Fortinet, Check Point) deployed on VMs. Use when you already standardise on a vendor and want the same policy on-prem and in cloud.

**Azure Firewall Manager** centralises firewall policy across VNets and Virtual WAN hubs. Don't confuse it with Azure Firewall itself — the manager is the policy plane, the firewall is the data plane.

AWS comparison: NAT Gateway ≈ AWS NAT Gateway directly; Azure Firewall ≈ AWS Network Firewall + some WAF behaviour.

## VNet peering

**VNet peering** connects two VNets at the fabric layer. Traffic between peered VNets flows over the Azure backbone — low latency, high throughput, no VPN or gateway hop.

Two flavours:

- **Regional peering** — both VNets in the same region.
- **Global peering** — VNets in different regions. Same model, slightly higher per-GB cost.

Three properties to remember:

- Peering is **non-transitive**. If A peers with B and B peers with C, A cannot reach C through B unless you turn on **gateway transit** (B's gateway/firewall is used as a hop) or set up a hub-and-spoke topology explicitly.
- Peering connections are **directional**. You must create the peering on both VNets, even though they refer to each other.
- Peering needs **non-overlapping address spaces**. This is the address-planning constraint that bites organisations who didn't plan early.

**Hub-and-spoke** is the standard enterprise topology: a central hub VNet holds the firewall, the VPN/ExpressRoute gateway, and DNS resolvers; each workload VNet is a spoke peered to the hub. Gateway transit lets the spokes reach on-prem via the hub's gateway without each spoke needing its own. The Cloud Adoption Framework landing zones from notebook 02 deploy exactly this shape automatically.

## VPN Gateway

**VPN Gateway** lets a VNet talk IPsec to on-prem or to remote users.

Two flavours:

- **Site-to-Site (S2S)** — IPsec tunnel between Azure and an on-prem VPN device. Bandwidth and reliability tied to the SKU (`VpnGw1` through `VpnGw5`, AZ-redundant variants). Used for hybrid networking without an ExpressRoute circuit.
- **Point-to-Site (P2S)** — individual user devices connect via OpenVPN or IKEv2; auth via certificate, RADIUS, or Entra ID. The work-from-home story.

**Active-active** mode runs two VPN Gateway instances simultaneously with two public IPs and two tunnels — both forward traffic. Pair it with a redundant on-prem device for higher availability. **Zone-redundant** SKUs (`VpnGw1AZ`+) spread the gateway across availability zones.

The throughput ceilings are real: `VpnGw1` does roughly 650 Mbps aggregate; `VpnGw5` does 10 Gbps. If you need more than that or sub-millisecond latency to on-prem, you've outgrown VPN — that's the ExpressRoute conversation.

## ExpressRoute

**ExpressRoute** is a private, dedicated layer-2/3 link between your on-prem network and Azure that *bypasses the public internet entirely*. You buy a **circuit** (with a chosen bandwidth, e.g. 200 Mbps to 100 Gbps) from a Microsoft connectivity partner — most large telcos and major colos. The circuit terminates at an ExpressRoute peering location (an Azure edge POP).

Two peering types over the circuit:

- **Private peering** — for connectivity to VNets via an ExpressRoute Gateway. The classic use.
- **Microsoft peering** — for connectivity to Microsoft 365, Dynamics 365, and Azure PaaS public endpoints over the private link. Rarely used since Private Link arrived.

Three features worth knowing:

- **Global Reach** — connects two ExpressRoute circuits in different regions so your on-prem sites talk to each other over Microsoft's backbone (cheaper and faster than running an MPLS between them).
- **FastPath** — bypasses the ExpressRoute Gateway for data-plane traffic, dropping latency a few ms. Worth turning on for high-throughput workloads.
- **ExpressRoute Direct** — port-level, no carrier in the middle. For organisations that already have presence in an Azure peering colo and want 10 or 100 Gbps without a partner.

ExpressRoute is the choice when you need deterministic latency, high bandwidth, or compliance constraints that forbid traversing the public internet. Costs reflect that — circuits plus port fees plus the gateway. Most enterprises run both ExpressRoute (primary) and S2S VPN (backup) for resilience.

## Virtual WAN

**Azure Virtual WAN** is the managed hub-and-spoke at planet scale. Instead of you building hubs in each region with peering, gateways, and firewalls glued together, Virtual WAN gives you a **virtual hub** per region with built-in:

- VNet connections (peering, managed)
- S2S VPN and P2S VPN endpoints
- ExpressRoute circuit attachments
- Azure Firewall integration
- Hub-to-hub global routing on Microsoft's backbone

Two SKUs: **Basic** (VPN-only), **Standard** (everything). Use Standard.

The pitch: a multinational with 30 branch sites and a dozen Azure regions sets up a Virtual WAN, attaches a hub per active region, connects each branch via VPN or ExpressRoute to the nearest hub, and lets Microsoft handle inter-hub routing. The alternative — building it by hand with peering and UDRs — is months of work and a permanent operational burden.

For small footprints (one or two regions, a handful of VNets), Virtual WAN is overkill — classic hub-and-spoke is simpler and cheaper. The crossover is somewhere around 5+ regions or 10+ branches.

## Service endpoints vs Private Link

Two ways to reach an Azure PaaS service (Storage, SQL, Key Vault, Cosmos) from a VNet without traversing the public internet — and the older one is the trap.

**Service endpoints** extend the VNet identity to the PaaS service. You enable the endpoint on a subnet and add the subnet's identity to the PaaS service's firewall allow-list. Traffic still goes to the *public* endpoint of the PaaS service — the IP is public — but the source is recognised as the VNet. Cheap, simple, but the PaaS service is still publicly reachable from anywhere else.

**Private Link / private endpoints** give the PaaS service a **private IP inside your VNet**. You connect to `mystorage.privatelink.blob.core.windows.net`, which resolves to the private IP. The public endpoint can be disabled entirely — the PaaS service is now genuinely private. This is the modern recommendation for any sensitive workload, and the model Microsoft is steering everyone toward.

The DNS detail to know: private endpoints need a **Private DNS zone** (`privatelink.blob.core.windows.net`) so VNet clients resolve to the private IP. Auto-integration with Azure Private DNS handles this if you let it; manual DNS deployments are where most private-endpoint outages come from.

AWS comparison: Service endpoints ≈ VPC gateway endpoints; Private Link ≈ AWS PrivateLink (the model is virtually identical).

In [ ]:
# Build a small hub-and-spoke from scratch.

RG=rg-net-demo
LOC=eastus
az group create --name $RG --location $LOC

# 1. Hub VNet with firewall and gateway subnets.
az network vnet create -g $RG -n vnet-hub \
  --address-prefix 10.0.0.0/16 \
  --subnet-name AzureFirewallSubnet --subnet-prefix 10.0.0.0/26
az network vnet subnet create -g $RG --vnet-name vnet-hub \
  --name GatewaySubnet --address-prefix 10.0.1.0/27

# 2. Spoke VNet with a workload subnet, NSG attached.
az network vnet create -g $RG -n vnet-spoke-app \
  --address-prefix 10.10.0.0/16 \
  --subnet-name snet-app --subnet-prefix 10.10.0.0/24
az network nsg create -g $RG -n nsg-app
az network nsg rule create -g $RG --nsg-name nsg-app -n allow-https-in \
  --priority 100 --access Allow --direction Inbound \
  --source-address-prefixes AzureLoadBalancer --destination-port-ranges 443 \
  --protocol Tcp
az network vnet subnet update -g $RG --vnet-name vnet-spoke-app --name snet-app \
  --network-security-group nsg-app

# 3. Peer hub <-> spoke (both directions).
az network vnet peering create -g $RG -n hub-to-spoke \
  --vnet-name vnet-hub --remote-vnet vnet-spoke-app \
  --allow-vnet-access --allow-forwarded-traffic --allow-gateway-transit
az network vnet peering create -g $RG -n spoke-to-hub \
  --vnet-name vnet-spoke-app --remote-vnet vnet-hub \
  --allow-vnet-access --allow-forwarded-traffic --use-remote-gateways

# 4. UDR on the spoke that forces all egress through a (hypothetical) firewall IP.
az network route-table create -g $RG -n rt-spoke-app
az network route-table route create -g $RG --route-table-name rt-spoke-app \
  --name to-fw --address-prefix 0.0.0.0/0 \
  --next-hop-type VirtualAppliance --next-hop-ip-address 10.0.0.4
az network vnet subnet update -g $RG --vnet-name vnet-spoke-app --name snet-app \
  --route-table rt-spoke-app

## Putting it together

A production network for a typical enterprise workload, top down:

1. **Address plan** — a single org-wide RFC 1918 allocation with `/16`s per VNet, no overlaps.
2. **Hub VNet** — Azure Firewall Premium in `AzureFirewallSubnet`, VPN or ExpressRoute Gateway in `GatewaySubnet`, optional Bastion in `AzureBastionSubnet`, Private DNS resolver for cross-VNet name resolution.
3. **Spoke VNets** — one per workload, peered to the hub, with workload subnets (NSGs attached, ASGs for app tiers) and delegated subnets for PaaS VNet integration.
4. **Egress** — UDR `0.0.0.0/0 → Azure Firewall` on every spoke subnet. NAT Gateway only when L7 inspection isn't needed.
5. **Hybrid** — ExpressRoute as primary, S2S VPN as backup; both terminate in the hub gateway.
6. **PaaS access** — Private Link endpoints in spoke subnets; Private DNS zones linked to hub and spoke VNets; service endpoints only on legacy services that don't support Private Link.
7. **Scale** — switch to Virtual WAN when you exceed ~5 regions or ~10 branch sites.

Get this shape right once and you build everything else on top of it for years. The mistakes are paid in pain — overlapping ranges that block peering, NSGs whose effective rules nobody can decode, asymmetric routes that drop production traffic at 3 a.m. The discipline up front is what avoids them.